In [ ]:
from math import log, sqrt, exp, erf
from datetime import datetime
import pandas as pd
import numpy as np
import json

def ncdf(x: float) -> float:
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))

XSP_df = pd.read_csv("./data-v0/XSP.csv")
IRX_df = pd.read_csv("./data-v0/IRX.csv")

XSP_prices = {item["Date"][0:10]: item["Close"] for item in XSP_df[["Date", "Close"]].to_dict(orient='index').values()}
IRX_prices = {item["Date"][0:10]: item["Close"] for item in IRX_df[["Date", "Close"]].to_dict(orient='index').values()}

with open("./data-v0/XSP-options.json", "r") as file:
    options = {o["ticker"]: o for o in json.load(file)}

bars = {}

with open("./data-v0/XSP-bars.jsonl", "r") as file:
    for line in file:
        bar = json.loads(line)
        bars[bar["ticker"]] = bar["res"]

measurements = []

for ticker, bar in bars.items():
    if "results" not in bar:
        continue

    for result in bar["results"]:
        option = options[ticker]
        date = datetime.utcfromtimestamp(result["t"] / 1000).strftime('%Y-%m-%d')

        m = {
            "ticker": ticker,
            "transaction_date": date,
            "expiration_date": option["expiration_date"],
            "strike_price": option["strike_price"],
            "option_price": result["vw"],
            "expiration_price": XSP_prices[option["expiration_date"]],
            "total_result": -result["vw"] + max(0, option["strike_price"] - XSP_prices[option["expiration_date"]]),
            "spot_price": XSP_prices[date],
            "strike_pct": option["strike_price"] / XSP_prices[date],
            "fed_rate": IRX_prices[date] / 100,
            "volume": result["v"]
        }

        trades = XSP_df[(XSP_df["Date"] >= m["transaction_date"]) & (XSP_df["Date"] <= m["expiration_date"])]
        
        trading_days = 252
        logret = np.log(trades["Close"]).diff().dropna()
        m["logret_len"] = len(logret)

        if (m["logret_len"] < 3):
            continue

        m["historical_sigma"] = float(logret.std(ddof=1) * np.sqrt(trading_days))

        transaction_date = datetime.strptime(m["transaction_date"], "%Y-%m-%d").date()
        expiration_date = datetime.strptime(m["expiration_date"], "%Y-%m-%d").date()
        m["days_to_expiry"] = (expiration_date - transaction_date).days
        m["T_years"] = m["days_to_expiry"] / 365.0

        vsqrtT = m["historical_sigma"] * sqrt(m["T_years"])
        d1 = (log(m["spot_price"] / m["strike_price"]) + (m["fed_rate"] + 0.5 * (m["historical_sigma"] ** 2)) * m["T_years"]) / vsqrtT
        d2 = d1 - vsqrtT

        m["bs_price"] = float(m["strike_price"] * exp(-m["fed_rate"] * m["T_years"]) * ncdf(-d2) - m["spot_price"] * ncdf(-d1))

        measurements.append(m)

In [ ]:
measurements[0]

In [15]:
pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', '{:.4f}'.format)

df = pd.DataFrame(measurements)
df = df[df["days_to_expiry"] > 150]
df = df.sort_values(by="strike_pct", ascending=False)
df["fair_pct"] = df["bs_price"] / df["option_price"]
df = df[["strike_pct", "days_to_expiry", "bs_price", "option_price", "fair_pct", "total_result"]]

df['strike_group'] = pd.qcut(df['strike_pct'], q=10, labels=False, duplicates='drop')
grouped_avg = df.groupby('strike_group').mean(numeric_only=True)
grouped_avg

,strike_pct,days_to_expiry,bs_price,option_price,fair_pct,total_result
strike_group,,,,,,
0,0.7086,217.3889,0.0356,2.4572,0.0128,-2.4572
1,0.8243,211.4444,1.4848,4.5521,0.3170,-4.5521
2,0.8430,207.8333,1.3055,4.0661,0.2420,-4.0661
3,0.8632,226.4706,1.5360,5.9928,0.2076,-5.9928
4,0.8907,187.9444,0.4413,5.1037,0.0872,-5.1037
5,0.9178,216.5556,3.2166,8.7943,0.3212,-8.7943
6,0.9576,254.8824,12.2465,13.9753,0.8323,-13.9753
7,0.9845,230.7222,13.5412,17.0689,0.7655,-17.0689
8,1.0018,231.2222,21.0672,22.9460,0.9314,-22.9460


In [ ]:
# how much APY good, how much bad